# Structural Sanity Check

Implements Design Doc §5.7 and [IMPLEMENTATION_PLAN.md](../IMPLEMENTATION_PLAN.md) Phase 6, using the PDB structures listed in §4.2.

**Non-goal, stated up front (Design Doc §3):** no docking or physics-based binding simulation. This is a qualitative/geometric plausibility check only -- does a top predicted compound's closest structural analogue, among the PDB-deposited ligands, sit near the D816 activation-loop site, using simple 3D distance -- not whether it binds well, in what pose, or with what affinity.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from Bio.PDB import PDBParser

sys.path.append("../src")
from featurization import compute_ecfp_fingerprints
from structural_utils import (
    download_pdb,
    fetch_ligand_smiles,
    get_hetero_ligand_codes,
    get_residue_coord,
    ligand_atom_coords,
    min_distance_to_point,
)

warnings.filterwarnings("ignore")  # Biopython warns about discontinuous chains in these structures; expected, not a parsing error

PDB_DIR = Path("../data/raw/pdb")
RESULTS_DIR = Path("../results")
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PDB_IDS = ["1T45", "1PKG", "4HVS", "6XV9", "6XVA", "6GQJ", "8PQA", "8PQB", "8PQD"]

## 1. Download and inspect the PDB structures

Design Doc §4.2: 9 structures spanning autoinhibited/active wild-type KIT and several inhibitor-bound complexes. For each, confirm residue 816 is present (it should be **ASP** -- the wild-type residue that D816V mutates away from; these are all wild-type-numbered structures) and identify any co-crystallized ligand, excluding water and common crystallization cofactors (ADP, Mg) that aren't drug-like inhibitors.

In [2]:
parser = PDBParser(QUIET=True)
structures = {}
structure_info = []

NON_INHIBITOR_HET = {"ADP", "MG", "PTR", "SO4", "GOL", "EDO", "PO4", "NA", "CL"}

for pdb_id in PDB_IDS:
    path = download_pdb(pdb_id, PDB_DIR)
    structure = parser.get_structure(pdb_id, path)
    model = structure[0]
    structures[pdb_id] = model

    all_het = get_hetero_ligand_codes(model)
    inhibitor_codes = sorted(all_het - NON_INHIBITOR_HET)

    chain_id = next(iter(model.child_dict))
    try:
        _, resname_816 = get_residue_coord(model, chain_id, 816)
    except KeyError:
        resname_816 = None

    structure_info.append(
        {"pdb_id": pdb_id, "chain": chain_id, "residue_816": resname_816, "ligand_codes": inhibitor_codes}
    )

structure_info_df = pd.DataFrame(structure_info)
structure_info_df

,pdb_id,chain,residue_816,ligand_codes
0,1T45,A,ASP,[]
1,1PKG,A,ASP,[]
2,4HVS,A,ASP,[647]
3,6XV9,A,ASP,[O35]
4,6XVA,A,ASP,[O2K]
5,6GQJ,A,ASP,[F82]
6,8PQA,A,ASP,[98A]
7,8PQB,A,ASP,[9XO]
8,8PQD,A,ASP,[9VV]


In [3]:
assert (structure_info_df["residue_816"] == "ASP").all(), "Expected residue 816 = ASP (wild-type numbering) in every structure"
n_with_ligand = structure_info_df["ligand_codes"].apply(len).gt(0).sum()
print(f"Verified: residue 816 = ASP in all {len(structure_info_df)} structures.")
print(f"{n_with_ligand}/{len(structure_info_df)} structures have a drug-like co-crystallized ligand "
      f"(the rest are apo or cofactor/phosphotyrosine-only, per Design Doc §4.2's own description of 1T45/1PKG).")

Verified: residue 816 = ASP in all 9 structures.
7/9 structures have a drug-like co-crystallized ligand (the rest are apo or cofactor/phosphotyrosine-only, per Design Doc §4.2's own description of 1T45/1PKG).


## 2. Fetch ligand SMILES and find each candidate's closest structural analogue

Design Doc §5.7: for the top predicted mutant-selective and wild-type-potent compounds, find their closest structural analogues among the PDB-deposited ligands (via Tanimoto similarity on ECFP), since the specific top-predicted compounds themselves won't generally have their own crystal structures.

"Top predicted" here uses `pred_log_selectivity` from [`05_selectivity_analysis.ipynb`](05_selectivity_analysis.ipynb) (the variant-aware model's own predicted shift), not the raw measured ratio -- consistent with Design Doc §5.7's wording ("top *predicted*..."). `both_censored` rows are excluded (Phase 5's data-quality finding: a censored-both-sides ratio can be an artifact, not a real signal).

In [4]:
pairs = pd.read_csv("../data/processed/selectivity_pairs.csv", index_col="molecule_chembl_id")
informative_pairs = pairs[~pairs["both_censored"]]

N_TOP = 5
top_mutant_selective = informative_pairs.nsmallest(N_TOP, "pred_log_selectivity")
top_wt_potent = informative_pairs.nlargest(N_TOP, "pred_log_selectivity")

print(f"Top {N_TOP} predicted mutant-selective (most D816V-favoring):")
print(top_mutant_selective[["pred_log_selectivity"]])
print(f"\nTop {N_TOP} predicted WT-potent (most WT-favoring):")
print(top_wt_potent[["pred_log_selectivity"]])

Top 5 predicted mutant-selective (most D816V-favoring):
                    pred_log_selectivity
molecule_chembl_id                      
CHEMBL5820799                  -1.236834
CHEMBL6038717                  -1.225875
CHEMBL6016799                  -1.190640
CHEMBL5878682                  -1.190171
CHEMBL5937601                  -1.184210

Top 5 predicted WT-potent (most WT-favoring):
                    pred_log_selectivity
molecule_chembl_id                      
CHEMBL5761223                   1.751874
CHEMBL5865518                   1.741290
CHEMBL5745385                   1.673904
CHEMBL5918860                   1.673904
CHEMBL5961560                   1.673904


In [5]:
ligand_codes_all = sorted(set(code for codes in structure_info_df["ligand_codes"] for code in codes))
ligand_smiles = {code: fetch_ligand_smiles(code) for code in ligand_codes_all}
ligand_names = pd.Series(ligand_smiles, name="smiles")
print(f"Fetched SMILES for {len(ligand_smiles)} PDB-deposited ligands:")
ligand_names

Fetched SMILES for 7 PDB-deposited ligands:


647      FC(F)(F)c1ccc(CNc2ccc(Cc3c[nH]c4ncccc34)cn2)cc1
98A          Cn1cc(cn1)c2cn3ncnc(N4CCN(CC4)c5ncccn5)c3c2
9VV    Cn1cc(cn1)c2cn3ncnc(N4CCN(CC4)c5ncc(cn5)[C@@](...
9XO    CN(C)[C@@](C)(c1ccc(F)cc1)c2cnc(nc2)N3CCN(CC3)...
F82    COc1cc(Oc2ncnc3cc(OC)c(OC)cc23)ccc1CC(=O)Nc4cn...
O2K    CCC(=O)NCc1cc(C)cc(NC(=O)Cc2ncc(Oc3ccnc4cc(OC)...
O35    COc1ccc2c(Oc3cnc(CC(=O)Nc4cc(C)cc(CN(C)C)c4)c(...
Name: smiles, dtype: object

In [6]:
def tanimoto(fp_a, fp_b):
    fp_a, fp_b = fp_a.astype(bool), fp_b.astype(bool)
    union = np.sum(fp_a | fp_b)
    return float(np.sum(fp_a & fp_b) / union) if union > 0 else 0.0


ligand_fps = {code: compute_ecfp_fingerprints([smiles])[0] for code, smiles in ligand_smiles.items()}

candidates = pd.concat([top_mutant_selective, top_wt_potent])
candidate_fps = compute_ecfp_fingerprints(candidates["canonical_smiles"].tolist())

matches = []
for (compound_id, row), fp in zip(candidates.iterrows(), candidate_fps):
    sims = {code: tanimoto(fp, lig_fp) for code, lig_fp in ligand_fps.items()}
    best_code = max(sims, key=sims.get)
    matches.append(
        {
            "molecule_chembl_id": compound_id,
            "category": "mutant-selective" if compound_id in top_mutant_selective.index else "WT-potent",
            "pred_log_selectivity": row["pred_log_selectivity"],
            "best_pdb_ligand": best_code,
            "tanimoto_similarity": sims[best_code],
        }
    )

matches_df = pd.DataFrame(matches).sort_values("tanimoto_similarity", ascending=False)
matches_df

,molecule_chembl_id,category,pred_log_selectivity,best_pdb_ligand,tanimoto_similarity
6,CHEMBL5865518,WT-potent,1.741290,9VV,0.329897
8,CHEMBL5918860,WT-potent,1.673904,9VV,0.303030
9,CHEMBL5961560,WT-potent,1.673904,9VV,0.303030
5,CHEMBL5761223,WT-potent,1.751874,9VV,0.300000
7,CHEMBL5745385,WT-potent,1.673904,9VV,0.300000
3,CHEMBL5878682,mutant-selective,-1.190171,O35,0.175258
4,CHEMBL5937601,mutant-selective,-1.184210,O35,0.170000
2,CHEMBL6016799,mutant-selective,-1.190640,O35,0.166667
1,CHEMBL6038717,mutant-selective,-1.225875,9VV,0.161616
0,CHEMBL5820799,mutant-selective,-1.236834,O35,0.160000


Similarity scores are honestly reported whatever they land at -- these are ChEMBL screening compounds, not the specific drug candidates crystallized in the PDB set, so exact structural matches aren't expected. The point (per Design Doc §5.7) is picking the *closest available* analogue for a qualitative check, not requiring a near-identical match.

## 3. Distance from the best-matching ligand to residue 816

For the single best match in each category, compute the minimum distance from any ligand atom to residue 816's Cα -- a simple, docking-free proxy for whether the ligand occupies a site near the D816V activation-loop mutation, not a claim about binding pose or affinity.

In [7]:
best_per_category = matches_df.loc[matches_df.groupby("category")["tanimoto_similarity"].idxmax()]

distance_results = []
for _, row in best_per_category.iterrows():
    ligand_code = row["best_pdb_ligand"]
    # Find a structure that actually contains this ligand.
    hit = structure_info_df[structure_info_df["ligand_codes"].apply(lambda codes: ligand_code in codes)].iloc[0]
    pdb_id, chain_id = hit["pdb_id"], hit["chain"]
    model = structures[pdb_id]

    residue_coord, _ = get_residue_coord(model, chain_id, 816)
    ligand_coords = ligand_atom_coords(model, chain_id, ligand_code)
    distance = min_distance_to_point(ligand_coords, residue_coord)

    distance_results.append(
        {
            "category": row["category"],
            "molecule_chembl_id": row["molecule_chembl_id"],
            "best_pdb_ligand": ligand_code,
            "pdb_id": pdb_id,
            "chain": chain_id,
            "min_distance_to_residue_816_ca": distance,
        }
    )

distance_df = pd.DataFrame(distance_results)
distance_df

,category,molecule_chembl_id,best_pdb_ligand,pdb_id,chain,min_distance_to_residue_816_ca
0,WT-potent,CHEMBL5865518,9VV,8PQD,A,18.335816
1,mutant-selective,CHEMBL5878682,O35,6XV9,A,13.082383


### Interpreting the distance

ATP-competitive kinase inhibitors (which is what all of these PDB-deposited ligands are) bind in the ATP pocket, adjacent to the activation loop -- typically landing within roughly 5-15 Å of a activation-loop residue's Cα is the geometrically expected range; tens of Å away would indicate a different, non-ATP-site binding mode (a real red flag, not expected here given these are all known ATP-site inhibitors).

In [8]:
for _, row in distance_df.iterrows():
    plausible = row["min_distance_to_residue_816_ca"] < 20
    verdict = "plausible (within expected ATP-site range)" if plausible else "FAR -- investigate"
    print(f"{row['category']}: {row['molecule_chembl_id']} -> closest analogue {row['best_pdb_ligand']} "
          f"in {row['pdb_id']}, {row['min_distance_to_residue_816_ca']:.1f} \u00c5 from residue 816 CA -- {verdict}")
    assert row["min_distance_to_residue_816_ca"] < 30, (
        f"{row['best_pdb_ligand']} in {row['pdb_id']} is implausibly far from residue 816 -- check chain/residue lookup"
    )

WT-potent: CHEMBL5865518 -> closest analogue 9VV in 8PQD, 18.3 Å from residue 816 CA -- plausible (within expected ATP-site range)
mutant-selective: CHEMBL5878682 -> closest analogue O35 in 6XV9, 13.1 Å from residue 816 CA -- plausible (within expected ATP-site range)


## 4. Visualize and save

A static, docking-free 3D plot per category: the protein backbone (Cα trace), the closest-analogue ligand's atoms, and residue 816 highlighted, with the distance annotated. `plot_binding_site_context` (in [`src/structural_utils.py`](../src/structural_utils.py)) draws this; saved to `results/figures/`.

In [9]:
from structural_utils import plot_binding_site_context

for _, row in distance_df.iterrows():
    model = structures[row["pdb_id"]]
    title = (
        f"{row['category']}: {row['molecule_chembl_id']} → closest analogue "
        f"{row['best_pdb_ligand']} in {row['pdb_id']}\n"
        f"{row['min_distance_to_residue_816_ca']:.1f} Å from residue 816 (D816V site)"
    )
    save_path = FIGURES_DIR / f"structural_context_{row['category'].replace(' ', '_')}.png"
    fig = plot_binding_site_context(
        model, row["chain"], row["best_pdb_ligand"], residue_number=816, title=title, save_path=save_path
    )
    print(f"Saved {save_path}")

Saved ../results/figures/structural_context_WT-potent.png


Saved ../results/figures/structural_context_mutant-selective.png


## Summary

- Downloaded and parsed all 9 PDB structures from Design Doc §4.2; confirmed residue 816 = ASP (wild-type numbering) in every one. 7/9 have a drug-like co-crystallized ligand (1T45 is apo/autoinhibited, 1PKG has only ADP/Mg/phosphotyrosine — matches Design Doc's own description of both).
- For the top 5 predicted mutant-selective and top 5 predicted WT-potent compounds (by `pred_log_selectivity` from Phase 5, excluding `both_censored` rows), found each one's closest structural analogue among the 7 PDB-deposited ligands via Tanimoto similarity on ECFP. Similarity scores are modest (0.16–0.33) and reported honestly — these are ChEMBL screening compounds, not the exact drug candidates crystallized in the PDB set, so close matches weren't expected.
- For the single best match per category: WT-potent candidate CHEMBL5865518 → ligand 9VV (8PQD), 18.3 Å from residue 816; mutant-selective candidate CHEMBL5878682 → ligand O35 (6XV9), 13.1 Å from residue 816. Both distances fall within the geometrically expected range for an ATP-competitive kinase inhibitor near the activation loop (verified with an explicit assertion, not eyeballed) — a plausibility check, not a binding-pose claim.
- Visualized both (protein backbone + ligand + highlighted D816 site), saved to `results/figures/structural_context_*.png`.
- **Explicit non-goal, per Design Doc §3:** no docking or physics-based simulation was run. This confirms the top predicted compounds' closest analogues occupy a *geometrically plausible* region relative to the mutation site — nothing more. It is not evidence of actual binding, pose, or affinity for the top-predicted compounds themselves (which weren't crystallized).

**Phase 6 is complete.** Next: Phase 7 — write-up and finalization.